# Metric 3 — LLM Judge
Run from project root: `jupyter notebook evaluation/visualisations/metric_3_judge.ipynb`

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

METRICS_DIR = Path('../../eval_results/metrics')
PATH = METRICS_DIR / 'metric_3.jsonl'

CONFIGS = [
    'api_no_glos', 'api_glos',
    'local_no_glos', 'local_glos',
    'finetuned_no_glos', 'finetuned_glos',
]
LABELS = {
    'api_no_glos':       'API (no glos)',
    'api_glos':          'API (glos)',
    'local_no_glos':     'Local (no glos)',
    'local_glos':        'Local (glos)',
    'finetuned_no_glos': 'Finetuned (no glos)',
    'finetuned_glos':    'Finetuned (glos)',
}
CRITERIA = ['mathematical_accuracy', 'terminology', 'grammar', 'naturalness', 'completeness']

# Build flat DataFrames: one per (part, criterion)
records = []
with open(PATH) as f:
    for line in f:
        rec = json.loads(line)
        idx = rec['idx']
        for ck in CONFIGS:
            val = rec.get(ck)
            if val is None:
                continue
            for part in ['problem', 'solution']:
                scores = val.get(part, {})
                row = {'idx': idx, 'config': ck, 'part': part}
                for cr in CRITERIA:
                    row[cr] = scores.get(cr)
                records.append(row)

df = pd.DataFrame(records)
df['overall'] = df[CRITERIA].mean(axis=1)
print(f'Loaded {len(df)} records ({df.idx.nunique()} examples x {df.config.nunique()} configs x 2 parts)')
df.head()

In [ ]:
# --- Overall avg score per config (problem + solution combined) ---
overall = df.groupby('config')['overall'].mean().reindex(CONFIGS)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar([LABELS[c] for c in CONFIGS], overall.values,
              color=sns.color_palette('muted', len(CONFIGS)))
ax.bar_label(bars, fmt='%.2f', padding=3)
ax.set_ylim(1, 5.5)
ax.set_ylabel('Avg score (1–5)')
ax.set_title('Metric 3 — LLM Judge: Overall Average Score per Config')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# --- Grouped bar: avg per criterion per config ---
crit_avgs = df.groupby('config')[CRITERIA].mean().reindex(CONFIGS)

x = np.arange(len(CRITERIA))
w = 0.13
fig, ax = plt.subplots(figsize=(14, 6))
colors = sns.color_palette('muted', len(CONFIGS))
for i, (ck, color) in enumerate(zip(CONFIGS, colors)):
    offset = (i - len(CONFIGS)/2 + 0.5) * w
    ax.bar(x + offset, crit_avgs.loc[ck], w, label=LABELS[ck], color=color)
ax.set_xticks(x)
ax.set_xticklabels([c.replace('_', ' ').title() for c in CRITERIA])
ax.set_ylim(1, 5.8)
ax.set_ylabel('Avg score (1–5)')
ax.set_title('Metric 3 — LLM Judge: Score per Criterion per Config')
ax.legend(fontsize=8, loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# --- Radar chart: one polygon per config ---
N = len(CRITERIA)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close polygon

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
colors = sns.color_palette('muted', len(CONFIGS))
for ck, color in zip(CONFIGS, colors):
    vals = crit_avgs.loc[ck].tolist()
    vals += vals[:1]
    ax.plot(angles, vals, linewidth=1.5, label=LABELS[ck], color=color)
    ax.fill(angles, vals, alpha=0.08, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels([c.replace('_', '\n').title() for c in CRITERIA], fontsize=9)
ax.set_ylim(1, 5)
ax.set_yticks([2, 3, 4, 5])
ax.set_title('Metric 3 — LLM Judge: Radar Chart', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# --- Heatmap: config x criterion ---
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, part in zip(axes, ['problem', 'solution']):
    heat = df[df.part == part].groupby('config')[CRITERIA].mean().reindex(CONFIGS)
    heat.index = [LABELS[c] for c in CONFIGS]
    heat.columns = [c.replace('_', ' ').title() for c in CRITERIA]
    sns.heatmap(heat, annot=True, fmt='.2f', cmap='YlGn', vmin=1, vmax=5,
                linewidths=0.5, ax=ax)
    ax.set_title(f'LLM Judge — {part.title()} scores')
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.suptitle('Metric 3 — LLM Judge Heatmap (1–5)', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Problem vs solution: overall avg per config ---
pv = df[df.part == 'problem'].groupby('config')['overall'].mean().reindex(CONFIGS)
sv = df[df.part == 'solution'].groupby('config')['overall'].mean().reindex(CONFIGS)

x = np.arange(len(CONFIGS))
w = 0.35
fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - w/2, pv, w, label='Problem',  color='steelblue')
b2 = ax.bar(x + w/2, sv, w, label='Solution', color='coral')
ax.bar_label(b1, fmt='%.2f', padding=3, fontsize=8)
ax.bar_label(b2, fmt='%.2f', padding=3, fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels([LABELS[c] for c in CONFIGS], rotation=20, ha='right')
ax.set_ylim(1, 5.8)
ax.set_ylabel('Overall avg score (1–5)')
ax.set_title('Metric 3 — LLM Judge: Problem vs Solution')
ax.legend()
plt.tight_layout()
plt.show()